# Construction de l'index FAISS

**Objectif :** découper les descriptions en chunks, les embedder avec `mistral-embed` et construire un index FAISS via LangChain.

Étapes :
1. Chargement des données traitées
2. Chunking — découpage de `longdescription_fr` en morceaux sémantiques
3. Création des Documents LangChain (chunk + métadonnées)
4. Construction de l'index FAISS par batch
5. Sauvegarde de l'index

In [ ]:
import os
import time
import pandas as pd
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document
from langchain_community.vectorstores import FAISS
from langchain_mistralai import MistralAIEmbeddings

df = pd.read_csv("../data/processed/events_clean.csv")
print(f"{len(df)} événements chargés")

## 2. Chunking

On découpe `longdescription_fr` en morceaux de 500 caractères maximum (avec 50 caractères de chevauchement).  
Le chevauchement permet d'éviter de couper une phrase en deux entre deux chunks consécutifs.

Chaque chunk devient un `Document` LangChain avec :
- `page_content` : le texte du chunk (ce qui sera embedé)
- `metadata` : les infos de l'événement (titre, dates, lieu, tarif, url)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

df = df.dropna(subset=["longdescription_fr"])

documents = []
for _, row in df.iterrows():
    chunks = splitter.split_text(row["longdescription_fr"])
    metadata = {
        "uid": row["uid"],
        "title": row["title_fr"],
        "conditions": row["conditions_fr"],
        "date_start": row["firstdate_begin"],
        "date_end": row["lastdate_end"],
        "location": f"{row['location_name']}, {row['location_address']}",
        "url": row["canonicalurl"],
    }
    for chunk in chunks:
        documents.append(Document(page_content=chunk, metadata=metadata))

print(f"{len(documents)} chunks créés pour {len(df)} événements")
print(f"Moyenne : {len(documents)/len(df):.1f} chunks par événement")

## 3. Initialisation du modèle d'embedding

On initialise `mistral-embed`. La dimension des vecteurs produits est 1024.

In [ ]:
embed_model = MistralAIEmbeddings(
    model="mistral-embed",
    mistral_api_key=os.getenv("MISTRAL_API_KEY")
)

## 4. Construction de l'index FAISS par batch

On utilise `FAISS.from_documents()` de LangChain qui gère automatiquement le stockage des métadonnées avec chaque vecteur.  
Pour respecter le rate limiting Mistral, on construit l'index par batch de 50 documents avec une pause entre chaque.

In [ ]:
batch_size = 50
faiss_store = None

for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    if faiss_store is None:
        faiss_store = FAISS.from_documents(batch, embed_model)
    else:
        faiss_store.add_documents(batch)
    print(f"{min(i + batch_size, len(documents))}/{len(documents)}", end="\r")
    time.sleep(1)

print(f"\nIndex construit : {faiss_store.index.ntotal} vecteurs")

os.makedirs("../data/faiss_index", exist_ok=True)
faiss_store.save_local("../data/faiss_index")
print("Index sauvegardé dans data/faiss_index/")